In [0]:
# Paso 1: Instalar las librerías necesarias
!pip install --upgrade langchain langchain-community langchain-databricks databricks-vectorsearch databricks-langchain
dbutils.library.restartPython()

In [0]:
pip list

In [0]:
from langchain.embeddings.base import Embeddings
from typing import List
import mlflow.deployments

class BluetabEmbeddingModel(Embeddings):
    def __init__(self, endpoint_name: str):
        self.endpoint_name = endpoint_name
        self.client = mlflow.deployments.get_deploy_client("databricks")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text)

    def _embed(self, text: str) -> List[float]:
        input_data = {
            "dataframe_split": {
                "columns": ["input"],
                "data": [[text]]
            }
        }
        response = self.client.predict(endpoint=self.endpoint_name, inputs=input_data)

        # Accede directamente al primer embedding de la lista
        return response["predictions"][0]


In [0]:
import os
from langchain_databricks import DatabricksEmbeddings, ChatDatabricks
# Importa DatabricksVectorSearch desde la librería correcta
from langchain_community.vectorstores import DatabricksVectorSearch
from databricks.vector_search.client import VectorSearchClient

from langchain.chains import RetrievalQA

# --- 1. Configuración Segura y Simplificada ---
# Es la mejor práctica configurar las credenciales como variables de entorno.
# LangChain las leerá automáticamente.
# Asegúrate de haber ejecutado estas líneas o tenerlas configuradas en tu entorno.
# os.environ['DATABRICKS_HOST'] = "https://dbc-ad7d5e59-0280.cloud.databricks.com/"
# os.environ['DATABRICKS_TOKEN'] = ""

# Nombres de tus endpoints y tu índice
INDEX_NAME = "bluetab.rag.docs_idx"
EMBEDDING_ENDPOINT = "simple_embbeding"
LLM_ENDPOINT = "flan_t5_base_model"
VECTOR_SEARCH_ENDPOINT = "doc_vector_endpoint"

# --- 2. Inicialización de Componentes ---

# Modelo de Embeddings (tu código ya era correcto)
embedding_model = BluetabEmbeddingModel(endpoint_name=EMBEDDING_ENDPOINT)

# Vector Store (forma simplificada y correcta)
# No necesitas crear un VectorSearchClient manualmente.
# La clase de LangChain solo necesita el nombre del endpoint del índice.
vs_client = VectorSearchClient()
vs_index = vs_client.get_index(
  endpoint_name=VECTOR_SEARCH_ENDPOINT,
  index_name=INDEX_NAME
)
vectorstore = DatabricksVectorSearch(
  index=vs_index,
  embedding=embedding_model,
  text_column="text"
)

# El retriever se crea a partir del vector store
retriever = vectorstore.as_retriever()

In [0]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# LLM para la generación de respuestas
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, max_tokens=200)

# --- 3. Creación del Retriever y la Cadena RAG ---

TEMPLATE = """You are an internal assistant chatbot for Bluetab employees. You answer questions related to Bluetab’s company information, products, technologies, employee guidelines, and internal processes. If the question is outside of these topics, politely decline to answer. If you don't know the answer, clearly state that you don't know and avoid inventing answers. If the question is about products or services not related to Bluetab, say so. Keep your answers clear and concise. Provide all answers only in Spanish.

Use the following pieces of context to answer the question at the end:
{context}
Pregunta: {question}
Respuesta:
"""

prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)



In [0]:
# --- 4. Ejecución de la Cadena ---

question = "¿Qué es bluetab?"
response = chain.invoke({"query": question})

print(response['result'])